# Mixed logit (MixL)

## Model

The time and cost coefficients vary across decision makers:

\[
\boldsymbol\beta^{R}_{nr}
=\boldsymbol\mu^{R}+\boldsymbol L\boldsymbol z_r,\qquad
\boldsymbol z_r\sim\mathcal N(\boldsymbol 0,\boldsymbol I).
\]

Conditional on draw \(r\), choice probabilities are logit. Integration is
approximated with \(R\) simulation draws:

\[
P_{nj}\approx\frac{1}{R}\sum_{r=1}^{R}
\frac{\exp(V_{njr})}{\sum_{k\in\mathcal C_n}\exp(V_{nkr})}.
\]

This example generates correlated random time and cost tastes, then estimates
their means, standard deviations, and Cholesky covariance term using
antithetic draws.

This notebook is self-contained and was executed in the repository's Office
validation environment. Install the released package with
`pip install torchdcm`, then select CPU or CUDA through the `device` argument.


In [1]:
import numpy as np
import torch

from IPython.display import HTML, display

from torchdcm import Beta, ChoiceDataset, MixedLogit, RandomCoefficient, UtilitySpec
from torchdcm.datasets import make_swissmetro_like
torch.set_default_dtype(torch.float64)
torch.manual_seed(7)
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"TorchDCM example running on {device}")


TorchDCM example running on cuda


In [2]:
rng = np.random.default_rng(14)
frame = make_swissmetro_like(n_obs=480, seed=14)
alternatives = np.array(["TRAIN", "SM", "CAR"])
time = frame[["time_train", "time_sm", "time_car"]].to_numpy()
cost = frame[["cost_train", "cost_sm", "cost_car"]].to_numpy()
available = frame[["avail_train", "avail_sm", "avail_car"]].to_numpy(bool)

sd_time, sd_cost, correlation = 0.008, 0.020, 0.35
covariance = np.array(
    [
        [sd_time**2, correlation * sd_time * sd_cost],
        [correlation * sd_time * sd_cost, sd_cost**2],
    ]
)
random_tastes = rng.multivariate_normal(
    mean=[-0.025, -0.070],
    cov=covariance,
    size=len(frame),
)
utility = (
    np.array([0.25, 0.0, 0.15])
    + random_tastes[:, [0]] * time
    + random_tastes[:, [1]] * cost
)
masked = np.where(available, utility, -np.inf)
probabilities = np.exp(masked - np.max(masked, axis=1, keepdims=True))
probabilities /= probabilities.sum(axis=1, keepdims=True)
frame["choice"] = [
    rng.choice(alternatives, p=row) for row in probabilities
]

data = ChoiceDataset.from_wide(
    frame,
    alternatives=alternatives.tolist(),
    choice="choice",
    variables={
        "time": {"TRAIN": "time_train", "SM": "time_sm", "CAR": "time_car"},
        "cost": {"TRAIN": "cost_train", "SM": "cost_sm", "CAR": "cost_car"},
    },
    availability={
        "TRAIN": "avail_train",
        "SM": "avail_sm",
        "CAR": "avail_car",
    },
    individual_id="person_id",
)
spec = UtilitySpec()
spec.utility(
    "TRAIN",
    Beta("ASC_TRAIN")
    + Beta("B_TIME", init=-0.02) * "time"
    + Beta("B_COST", init=-0.05) * "cost",
)
spec.utility(
    "SM",
    Beta("B_TIME", init=-0.02) * "time"
    + Beta("B_COST", init=-0.05) * "cost",
)
spec.utility(
    "CAR",
    Beta("ASC_CAR")
    + Beta("B_TIME", init=-0.02) * "time"
    + Beta("B_COST", init=-0.05) * "cost",
)


UtilitySpec(utilities=OrderedDict({'TRAIN': Expression(terms=[Term(parameter=Beta(name='ASC_TRAIN', init=0.0, fixed=False), variable=None, multiplier=1.0), Term(parameter=Beta(name='B_TIME', init=-0.02, fixed=False), variable='time', multiplier=1.0), Term(parameter=Beta(name='B_COST', init=-0.05, fixed=False), variable='cost', multiplier=1.0)]), 'SM': Expression(terms=[Term(parameter=Beta(name='B_TIME', init=-0.02, fixed=False), variable='time', multiplier=1.0), Term(parameter=Beta(name='B_COST', init=-0.05, fixed=False), variable='cost', multiplier=1.0)]), 'CAR': Expression(terms=[Term(parameter=Beta(name='ASC_CAR', init=0.0, fixed=False), variable=None, multiplier=1.0), Term(parameter=Beta(name='B_TIME', init=-0.02, fixed=False), variable='time', multiplier=1.0), Term(parameter=Beta(name='B_COST', init=-0.05, fixed=False), variable='cost', multiplier=1.0)])}))

In [3]:
model = MixedLogit(
    spec,
    [
        RandomCoefficient("B_TIME", sigma_init=0.006),
        RandomCoefficient("B_COST", sigma_init=0.015),
    ],
    n_draws=32,
    seed=14,
    antithetic=True,
    panel=False,
    correlated=True,
    device=device,
    max_iter=80,
)
result = model.fit(data)
display(HTML(result.report().to_html()))


Model family,MixedLogit
Estimation objective,Maximum simulated log likelihood
TorchDCM version,0.1.0
PyTorch version,2.12.1+cu130
Python version,3.12.13
Operating system,Linux-6.17.0-35-generic-x86_64-with-glibc2.39
Device,cuda
Tensor dtype,float64
Optimizer,torch.optim.LBFGS
Maximum iterations,80
Line search,strong_wolfe
